# Justice League Stellar Merger History

## Charlotte Christensen, October 23 2025

This jupyter notebook uses the output Anna Wright's code as modified by Nithun Selva (/home/christenc/Code/python/NithunSelva_startrace/RomZoomSHAnalysisScripts) to find the ID of the last satellite to merger into the main halo branch

In [1]:
import pynbody as pb
import numpy as np
import pandas as pd
import glob
import os
import h5py
import tangos

# For importing modules
import importlib.util
import sys
from pathlib import Path

import tangos.examples.mergers as mergers

In [2]:
#After running .bashrc to set up tangos environment, run the server with:
# tangos serve
# in the terminal


# can supposedly be done from within jupyter notebook with:
#!tangos serve

In [3]:
# Import modules

base_path = '/home/christenc/Code/python/NithunSelva_startrace'

for root, dirs, files in os.walk(base_path):
    if root not in sys.path:
        sys.path.append(root)

In [8]:
simpath = '/data/REPOSITORY/romulus_zooms'
os.environ['TANGOS_SIMULATION_FOLDER'] = simpath
# os.environ['TANGOS_DB_CONNECTION'] = simpath + 'rom25_dwarf_zooms.db'
tangos.core.init_db(simpath + '/rom25_dwarf_zooms.db')
#tangos.all_simulations()

# Completed 442, 492, 515, 523, 544, 552, 568, 569,  597, 613, 634, 642, 656, 718, 719, 741, 749, 761, 886, 916, 
# 918, 968, 977
simroot = 'r614' # Error on 431; 571, 614, 615, 707
simname = simroot + '.romulus25.3072g1HsbBH'

In [9]:
# Read in the unique halo ids assigned for all star particles
sh_path = '/home/awright/dwarf_stellar_halos/'
write_path = '/home/christenc/Code/Datafiles/stellar_halos/'

with h5py.File(sh_path+'/'+simroot+'/'+simroot+'_allhalostardata.h5','r') as f:
    hostids = f['host_IDs'].asstr()[:] # unique host IDs
    partids = f['particle_IDs'][:] # iords
    pct = f['particle_creation_times'][:] # formation times
    ph = f['particle_hosts'][:] # local host IDs (i.e., host at formation time)
    pp = f['particle_positions'][:] # position at formation time
    tsloc = f['timestep_location'][:] # snapshot where star particle first appears
uIDs = np.unique(hostids)

In [10]:
# Read in tangos database
sim = tangos.get_simulation(simname)

# Determine the main progenitor of the main halo
mh_prog,mh_dbid = sim[-1][1].calculate_for_progenitors('halo_number()','dbid()')
mh_halos = [tangos.get_halo(dbid) for dbid in mh_dbid]

#for mh_halo in mh_halos:
#    print(f"Main Halo Progenitor: {mh_halo}")

In [11]:
uIDs

array(['0096_-1', '0096_-2', '0096_-3', '0096_-4', '0192_-1', '0192_-2',
       '0192_14', '0192_19', '0192_25', '0288_-1', '0288_-2', '0288_15',
       '0384_3', '0576_16', '0576_7', '0672_35', '0672_6', '0768_3',
       '0768_8', '1056_6', '4096_1', '4096_166', '4096_18', '4096_22',
       '4096_3'], dtype=object)

In [12]:
# For each of the unique halo ids, 
halo_dict = {}
for uID in uIDs:
    # uID = '0384_10'
    sat_ts = uID.split('_')[0]
    sat_hid = int(uID.split('_')[1])
    halo_dict[uID] = uID
    if sat_hid <= 0:
        print('Skipping',uID)
        continue
    db_halo = tangos.get_halo(f"{simname}/%{sat_ts}/halo_{sat_hid}")
    # trace forward to get the merger history
    sat_desc,sat_dbid = db_halo.calculate_for_descendants('halo_number()','dbid()')   
    # sat_desc = sat_desc[::-1]  
    # sat_time = sat_time[::-1]
    sat_halos = [tangos.get_halo(dbid) for dbid in sat_dbid][::-1] # reverse
    # print(f"Satellite Halo: {sat_halos}, Total Descendants: {len(sat_desc)}")               

    # Identify the first most recent time when the descendent of the satellite
    # is not the main halo main progenitor
    for i in range(len(sat_halos)):
        if sat_halos[i] != mh_halos[i]:
            # sat_id = f'{sat_halos[i].timestep.extension[-4:]}_{sat_halos[i].halo_number()}'
            mh_id = f'{sat_halos[i].timestep.extension[-4:]}_{sat_halos[i].halo_number}'
            if mh_id in uIDs:
                print(f"Merger Identified at {sat_halos[i].timestep.extension[-6:]}: Sat Halo {uID} merges into Main Halo Progenitor {mh_id}")
            else:
                print(f"Merger into new halo at {sat_halos[i].timestep.extension[-6:]}: Sat Halo {uID} merges into Main Halo Progenitor {mh_id}, but not in uIDs")
            # if i > 0:
            #     print(f"Merger Identified at {sat_halos[i-1].timestep.extension[-6:]}: Sat Halo {sat_halos[i-1].halo_number} merges into Main Halo Progenitor {mh_halos[i-1].halo_number}")
            halo_dict[uID] = mh_id
            break

    #if i != 0:
    #    print(i, sat_desc[i-1], mh_prog[i-1], sat_time[i-1])
    # print(i, sat_desc[i], mh_prog[i], sat_time[i])

Skipping 0096_-1
Skipping 0096_-2
Skipping 0096_-3
Skipping 0096_-4
Skipping 0192_-1
Skipping 0192_-2
Merger Identified at 000192: Sat Halo 0192_14 merges into Main Halo Progenitor 0192_14
Merger into new halo at 000480: Sat Halo 0192_19 merges into Main Halo Progenitor 0480_4, but not in uIDs
Merger Identified at 000192: Sat Halo 0192_25 merges into Main Halo Progenitor 0192_25
Skipping 0288_-1
Skipping 0288_-2
Merger Identified at 000288: Sat Halo 0288_15 merges into Main Halo Progenitor 0288_15
Merger Identified at 000384: Sat Halo 0384_3 merges into Main Halo Progenitor 0384_3
Merger Identified at 000576: Sat Halo 0576_16 merges into Main Halo Progenitor 0576_16
Merger Identified at 000576: Sat Halo 0576_7 merges into Main Halo Progenitor 0576_7
Merger Identified at 000672: Sat Halo 0672_35 merges into Main Halo Progenitor 0672_35
Merger Identified at 000672: Sat Halo 0672_6 merges into Main Halo Progenitor 0672_6
Merger Identified at 000768: Sat Halo 0768_3 merges into Main Halo P

In [14]:
halo_dict

{'0096_-1': '0096_-1',
 '0096_-2': '0096_-2',
 '0096_-3': '0096_-3',
 '0096_-4': '0096_-4',
 '0192_-1': '0192_-1',
 '0192_-2': '0192_-2',
 '0192_14': '0192_14',
 '0192_19': '0480_4',
 '0192_25': '0192_25',
 '0288_-1': '0288_-1',
 '0288_-2': '0288_-2',
 '0288_15': '0288_15',
 '0384_3': '0384_3',
 '0576_16': '0576_16',
 '0576_7': '0576_7',
 '0672_35': '0672_35',
 '0672_6': '0672_6',
 '0768_3': '0768_3',
 '0768_8': '0768_8',
 '1056_6': '1056_6',
 '4096_1': '4096_1',
 '4096_166': '4096_166',
 '4096_18': '4096_18',
 '4096_22': '4096_22',
 '4096_3': '4096_3'}

In [51]:
if simroot == 'r442':
    halo_dict['0672_5'] = '0672_5' # Manually uncombine MM 442
    halo_dict['0576_-1'] = '1248_2' # Manually combine ghost halo for MM 442
elif simroot == 'r492':
    halo_dict['0273_5'] = '1268_2' # MM 492
    halo_dict['1056_7'] = '1056_7' # MM 492. # Double check that this right
elif simroot == 'r515':
    halo_dict['0192_-2'] = '1440_2' # MM 515
    halo_dict['0288_-1'] = '1440_2' # MM 515
    halo_dict['0480_5'] = '0672_44'
    halo_dict['0480_6'] = '0672_44'
elif simroot == 'r523':
    halo_dict['0480_13'] = '0972_2'
    halo_dict['0576_10'] = '0972_2'
    halo_dict['0672_3'] = '1920_9'
elif simroot == 'r544':
    halo_dict['0288_-4'] = '0288_-1'     # MM 544
    halo_dict['0288_-2'] = '0288_-1'     # MM 544
elif simroot == 'r552':
    print("nothing to manually adjust for r552")
    #halo_dict['0192_-4'] = '1344_6'     # MM 552
elif simroot == 'r568':
    print("nothing to manually adjust for r568")
elif simroot == 'r569':
    print("nothing to manually adjust for r569")





In [15]:
hostids2 = [halo_dict[hid] for hid in hostids]

In [16]:
np.unique(hostids2)

array(['0096_-1', '0096_-2', '0096_-3', '0096_-4', '0192_-1', '0192_-2',
       '0192_14', '0192_25', '0288_-1', '0288_-2', '0288_15', '0384_3',
       '0480_4', '0576_16', '0576_7', '0672_35', '0672_6', '0768_3',
       '0768_8', '1056_6', '4096_1', '4096_166', '4096_18', '4096_22',
       '4096_3'], dtype='<U8')

In [17]:
os.makedirs(os.path.join(write_path, simroot), exist_ok=True)
with h5py.File(write_path+'/'+simroot+'/'+simroot+'_allhalostardata_consolidated2.h5','w') as f:
    f.create_dataset('particle_IDs', data=partids) # iords
    f.create_dataset('particle_positions', data=pp) # position at formation time
    f.create_dataset('particle_creation_times', data=pct) # time of formation
    f.create_dataset('timestep_location', data=tsloc) # snapshot where star first appears
    f.create_dataset('particle_hosts', data=ph) # host at that snapshot
    f.create_dataset('host_IDs', data=hostids2) # unique ID of host

In [55]:
#!tangos serve

In [56]:
#desc,fid,phid = sim[int(step)][int(halo)].calculate_for_descendants('halo_number()','finder_id()','type()')
#proj,fid,phid = sim[int(unis)][unih].calculate_for_progenitors('halo_number()','finder_id()','type()')